# Description Task — Visualization
Load pre-computed evaluation results from `output/description_eval_results.json` and visualize.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from pathlib import Path

matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 120})

# Load pre-computed results
RESULTS_PATH = Path('..') / 'output' / 'description_eval_results.json'
with open(RESULTS_PATH) as f:
    all_results = json.load(f)

df = pd.DataFrame(all_results)
print(f'Loaded {len(df)} entries from {RESULTS_PATH}')
df

In [ ]:
# Summary table (formatted)
metrics = ['bleu', 'rouge_l', 'meteor', 'bertscore_r', 'bertscore_f1', 'sbert_similarity', 'clipscore']
display_cols = ['model', 'condition', 'n_samples'] + metrics
df_display = df[display_cols].copy()
for col in metrics:
    df_display[col] = df_display[col].map(lambda x: f'{x:.4f}')
df_display

In [ ]:
# Grouped bar chart: length-sensitive metrics (top) vs length-invariant metrics (bottom)
models = df['model'].unique()
conditions = ['original', 'weather', 'night', 'small']
colors = {'original': '#2196F3', 'weather': '#4CAF50', 'night': '#9C27B0', 'small': '#FF9800'}

# Row 1: length-sensitive metrics
metrics_sensitive = ['bleu', 'rouge_l', 'meteor', 'bertscore_f1']
labels_sensitive = ['BLEU', 'ROUGE-L', 'METEOR', 'BERTScore-F1']

# Row 2: length-invariant metrics (fairer for augmentation)
metrics_invariant = ['bertscore_r', 'sbert_similarity', 'clipscore']
labels_invariant = ['BERTScore-Recall', 'SBERT Similarity', 'CLIPScore']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Top row: length-sensitive
for ax, metric, label in zip(axes[0], metrics_sensitive, labels_sensitive):
    x = np.arange(len(models))
    width = 0.18
    for i, cond in enumerate(conditions):
        vals = [df[(df['model'] == m) & (df['condition'] == cond)][metric].values[0]
                if len(df[(df['model'] == m) & (df['condition'] == cond)]) > 0 else 0
                for m in models]
        bars = ax.bar(x + i * width, vals, width, label=cond, color=colors[cond])
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)
    ax.set_title(label, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([m.replace('_', '\n') for m in models], fontsize=8)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.25)

axes[0][0].legend(loc='upper right', fontsize=8)

# Bottom row: length-invariant
for ax, metric, label in zip(axes[1], metrics_invariant, labels_invariant):
    x = np.arange(len(models))
    width = 0.18
    for i, cond in enumerate(conditions):
        vals = [df[(df['model'] == m) & (df['condition'] == cond)][metric].values[0]
                if len(df[(df['model'] == m) & (df['condition'] == cond)]) > 0 else 0
                for m in models]
        bars = ax.bar(x + i * width, vals, width, label=cond, color=colors[cond])
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)
    ax.set_title(label, fontweight='bold', color='#1565C0')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([m.replace('_', '\n') for m in models], fontsize=8)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

axes[1][-1].set_visible(False)  # hide empty 4th subplot in bottom row

fig.text(0.5, 1.02, 'Length-Sensitive Metrics (penalize extra augmentation descriptions)',
         ha='center', fontsize=12, color='gray', transform=axes[0][1].transAxes)
fig.text(0.5, 1.02, 'Length-Invariant Metrics (fairer for augmentation comparison)',
         ha='center', fontsize=12, color='#1565C0', fontweight='bold', transform=axes[1][1].transAxes)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: model x condition for ALL metrics (including new ones)
fig, axes = plt.subplots(2, 4, figsize=(24, 8))

all_metrics = ['bleu', 'rouge_l', 'meteor', 'bertscore_f1', 'bertscore_r', 'sbert_similarity', 'clipscore']
all_labels = ['BLEU', 'ROUGE-L', 'METEOR', 'BERTScore-F1', 'BERTScore-Recall*', 'SBERT Similarity*', 'CLIPScore*']

flat_axes = axes.flatten()
for idx, (ax, metric, label) in enumerate(zip(flat_axes, all_metrics, all_labels)):
    pivot = df.pivot(index='model', columns='condition', values=metric)
    pivot = pivot.reindex(columns=conditions)
    
    im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(conditions)))
    ax.set_xticklabels(conditions, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    is_invariant = '*' in label
    ax.set_title(label, fontweight='bold', fontsize=11,
                 color='#1565C0' if is_invariant else 'black')
    
    for i in range(len(pivot.index)):
        for j in range(len(conditions)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8,
                    color='white' if val > pivot.values.mean() + pivot.values.std() else 'black')

flat_axes[-1].set_visible(False)  # hide empty 8th subplot
fig.suptitle('Description Metrics Heatmap (* = length-invariant)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Radar chart: model profiles on original condition (all metrics, normalized by max)
from math import pi

radar_metrics = ['bleu', 'rouge_l', 'meteor', 'bertscore_r', 'sbert_similarity', 'clipscore']
radar_labels = ['BLEU', 'ROUGE-L', 'METEOR', 'BERT-Recall', 'SBERT-Sim', 'CLIPScore']

df_orig = df[df['condition'] == 'original']

# Auto-assign distinct colors for all models
_cmap = plt.cm.tab10
model_colors_map = {m: _cmap(i / max(len(df_orig) - 1, 1)) for i, m in enumerate(df_orig['model'].unique())}

# Normalize by max so radar shape is comparable across different scales
max_vals = {m: df_orig[m].max() for m in radar_metrics}

angles = np.linspace(0, 2 * pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for mi, (_, row) in enumerate(df_orig.iterrows()):
    raw_vals = [row[m] for m in radar_metrics]
    norm_vals = [row[m] / max_vals[m] * 100 if max_vals[m] > 0 else 0 for m in radar_metrics]
    norm_vals += norm_vals[:1]
    raw_vals += raw_vals[:1]
    color = model_colors_map[row['model']]
    ax.plot(angles, norm_vals, 'o-', linewidth=2, label=row['model'], color=color, markersize=6)
    ax.fill(angles, norm_vals, alpha=0.1, color=color)
    
    for j in range(len(radar_metrics)):
        offset = 10 if mi % 2 == 0 else -10
        ax.annotate(f'{raw_vals[j]:.3f}',
                    xy=(angles[j], norm_vals[j]),
                    fontsize=7, fontweight='bold', color=color,
                    textcoords='offset points', xytext=(0, offset),
                    ha='center', va='center')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_ylim(0, 110)
ax.set_yticks([25, 50, 75, 100])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='gray')
ax.set_title('Model Profiles — Original Condition\n(normalized by best, labels = raw values)', fontsize=13, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Degradation analysis: % drop from original — length-sensitive vs length-invariant
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

aug_conditions = ['weather', 'night', 'small']

# Row 1: length-sensitive
sensitive_metrics = ['bleu', 'rouge_l', 'meteor', 'bertscore_f1']
sensitive_labels = ['BLEU', 'ROUGE-L', 'METEOR', 'BERT-F1']

# Row 2: length-invariant
invariant_metrics = ['bertscore_r', 'sbert_similarity', 'clipscore']
invariant_labels = ['BERT-Recall', 'SBERT-Sim', 'CLIPScore']

for row_idx, (row_metrics, row_labels, title_suffix) in enumerate([
    (sensitive_metrics, sensitive_labels, 'Length-Sensitive'),
    (invariant_metrics, invariant_labels, 'Length-Invariant'),
]):
    for col_idx, model in enumerate(models):
        ax = axes[row_idx][col_idx]
        orig = df[(df['model'] == model) & (df['condition'] == 'original')].iloc[0]
        drops = {}
        for cond in aug_conditions:
            row = df[(df['model'] == model) & (df['condition'] == cond)]
            if len(row) == 0:
                continue
            row = row.iloc[0]
            drops[cond] = [(orig[m] - row[m]) / orig[m] * 100 if orig[m] != 0 else 0 for m in row_metrics]
        
        x = np.arange(len(row_metrics))
        width = 0.25
        for i, cond in enumerate(aug_conditions):
            if cond in drops:
                ax.bar(x + i * width, drops[cond], width, label=cond, color=colors[cond])
        
        ax.set_ylabel('% Drop from Original')
        ax.set_title(f'{model.replace("_", " ")}\n({title_suffix})',
                     fontweight='bold', fontsize=9,
                     color='#1565C0' if row_idx == 1 else 'black')
        ax.set_xticks(x + width)
        ax.set_xticklabels(row_labels, fontsize=8)
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
        if col_idx == 0:
            ax.legend(fontsize=8)

fig.suptitle('Performance Degradation: Length-Sensitive (top) vs Length-Invariant (bottom)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()